# BETO — Nested CV 5×3 con seguimiento y recuperación

Mismo experimento de BETO por variante dialectal (MX, ES, CU) del notebook anterior, pero con una salida breve y un panel de seguimiento que se actualiza durante la ejecución.

- En **modo prueba**: MX, 2 Outer, 2 Inner, 2 trials.
- En **modo completo**: MX/ES/CU, 5 Outer, 3 Inner, 10 trials (450 fine-tunings internos y 15 refits).
- `BASE` y `SOCIAL`, tasa de aprendizaje, batch, épocas, weight decay y pérdida siguen compitiendo **dentro del Inner CV** por F1-Macro.
- Los pesos del modelo nunca se heredan entre folds o trials.
- **No se carga el test oficial** hasta habilitar su sección expresamente.

**Cambios frente a la versión anterior:** se desactivaron las barras y tablas repetitivas de Trainer; hay un solo panel vivo de progreso, un archivo `monitor.log`, un estudio Optuna SQLite por Outer Fold y un CSV de resultados parciales por variante. Los trials completos se recuperan desde SQLite y los Outer Folds completados se omiten al reanudar. Un entrenamiento interrumpido **a mitad de un Inner Fold** sí deberá repetirse: no es posible reanudar desde la última operación de GPU sin guardar checkpoints adicionales.


## 1. Dependencias

El entorno necesita:

```powershell
py -m uv add torch transformers accelerate optuna scikit-learn pandas numpy
```

o el equivalente con `pip`.

`Trainer` requiere `accelerate`.

> El notebook utiliza padding dinámico y `max_length=128`. El diagnóstico previo mostró que prácticamente todos los tweets quedan dentro de ese límite.


In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.0+cu130
CUDA: 13.0
GPU disponible: True
GPU: NVIDIA RTX 6000 Ada Generation


In [2]:
import os
import gc
import json
import math
import random
import inspect
import tempfile
import warnings
import time
import pickle
import hashlib
import html
from datetime import datetime
from contextlib import redirect_stdout, redirect_stderr
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import optuna

from torch.utils.data import Dataset, Subset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report,
)
from sklearn.utils.class_weight import compute_class_weight

import transformers
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    TrainerCallback,
    set_seed,
)

warnings.filterwarnings("ignore", category=FutureWarning)
transformers.logging.set_verbosity_error()
optuna.logging.set_verbosity(optuna.logging.WARNING)
from transformers.trainer_callback import PrinterCallback, ProgressCallback
from IPython.display import display, HTML

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Optuna:", optuna.__version__)
print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM total (GB):",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / (1024**3),
            2,
        ),
    )
    print(
        "BF16 soportado:",
        torch.cuda.is_bf16_supported(),
    )


ModuleNotFoundError: No module named 'torch'

## 2. Configuración experimental

### Modo de ejecución

Primero ejecuta con:

```python
MODO = "prueba"
```

para verificar que todo el pipeline corre correctamente.

Después cambia **solo** a:

```python
MODO = "completo"
```

para el experimento real 5×3 con 10 trials.

El test oficial permanece deshabilitado en ambos modos hasta que se congele la configuración final.


In [ ]:
# ============================================================
# CONFIGURACIÓN PRINCIPAL
# ============================================================

MODEL_NAME = "dccuchile/bert-base-spanish-wwm-cased"

DATA_DIR = Path("../data/transformers")
RESULTS_DIR = Path("../data/resultados_beto")
TMP_DIR = RESULTS_DIR / "tmp"
# Separar resultados de prueba y completos; NO mezclar experimentos.
PROGRESS_DIR = RESULTS_DIR / "seguimiento"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
TMP_DIR.mkdir(parents=True, exist_ok=True)

TEXT_COLUMNS = {
    "BASE": "MESSAGE_TRANSFORMER_BASE",
    "SOCIAL": "MESSAGE_TRANSFORMER_SOCIAL",
}

LABEL_COL = "IS_IRONIC"
MAX_LENGTH = 128
RANDOM_STATE = 42

# ------------------------------------------------------------
# PRIMERO: "prueba"
# DESPUÉS: "completo"
# ------------------------------------------------------------
MODO = "prueba"

if MODO == "prueba":
    VARIANTES = ["mx"]
    N_OUTER = 2
    N_INNER = 2
    N_TRIALS = 2
elif MODO == "completo":
    VARIANTES = ["mx", "es", "cu"]
    N_OUTER = 5
    N_INNER = 3
    N_TRIALS = 10
else:
    raise ValueError(
        "MODO debe ser 'prueba' o 'completo'."
    )

# Optimización final se ejecuta después de revisar Nested CV.
EJECUTAR_OPTIMIZACION_FINAL = False

# Test oficial: mantener FALSE hasta congelar metodología/configuración.
EJECUTAR_TEST_OFICIAL = False

SEMILLAS_FINALES = [42, 52, 62, 72, 82]

# Actualizar el mismo panel cada cierto tiempo; no crear cientos de outputs.
HEARTBEAT_SEGUNDOS = 30

# Mixed precision
USE_BF16 = (
    torch.cuda.is_available()
    and torch.cuda.is_bf16_supported()
)
USE_FP16 = (
    torch.cuda.is_available()
    and not USE_BF16
)

if torch.cuda.is_available():
    torch.set_float32_matmul_precision("high")

print("MODO:", MODO)
print("Variantes:", VARIANTES)
print("Outer:", N_OUTER)
print("Inner:", N_INNER)
print("Trials:", N_TRIALS)
print("BF16:", USE_BF16)
print("FP16:", USE_FP16)

fits_aprox = (
    len(VARIANTES)
    * N_OUTER
    * N_TRIALS
    * N_INNER
)

print(
    "\nFine-tunings internos aproximados:",
    fits_aprox,
)

# La carpeta definitiva se define DESPUÉS de conocer MODO.
RUN_DIR = PROGRESS_DIR / MODO
RUN_DIR.mkdir(parents=True, exist_ok=True)


NameError: name 'Path' is not defined

## 3. Reproducibilidad

La semilla se fija en Python, NumPy, PyTorch y Transformers.

Esto mejora la reproducibilidad, aunque en GPU no siempre es posible garantizar resultados bit-a-bit idénticos entre hardware/versiones.


In [ ]:
def fijar_semilla(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    set_seed(seed)


def liberar_memoria():
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


fijar_semilla(RANDOM_STATE)


## 4. Carga del train Transformer

Durante Nested CV se cargan **únicamente**:

```text
train_transformers_mx.csv
train_transformers_es.csv
train_transformers_cu.csv
```

El test oficial no se carga aquí.


In [ ]:
def cargar_train_variante(variante):
    ruta = (
        DATA_DIR
        / f"train_transformers_{variante}.csv"
    )

    if not ruta.exists():
        raise FileNotFoundError(
            f"No existe {ruta}. "
            "Ejecuta primero el notebook de preprocessing DL final."
        )

    df = pd.read_csv(ruta)

    requeridas = [
        LABEL_COL,
        TEXT_COLUMNS["BASE"],
        TEXT_COLUMNS["SOCIAL"],
    ]

    faltantes = [
        col for col in requeridas
        if col not in df.columns
    ]

    if faltantes:
        raise ValueError(
            f"{variante}: faltan columnas {faltantes}"
        )

    df[LABEL_COL] = (
        df[LABEL_COL]
        .astype(int)
    )

    for col in TEXT_COLUMNS.values():
        df[col] = (
            df[col]
            .fillna("")
            .astype(str)
        )

    return df.reset_index(drop=True)


for variante in VARIANTES:
    df_tmp = cargar_train_variante(variante)

    print(
        f"{variante.upper()}: "
        f"{len(df_tmp)} ejemplos | "
        f"{df_tmp[LABEL_COL].value_counts().to_dict()}"
    )


## 5. Tokenizer y padding dinámico

Se utiliza el tokenizer oficial del checkpoint BETO.

- `max_length = 128`
- truncamiento habilitado
- **sin padding al tokenizar**
- padding dinámico por batch con `DataCollatorWithPadding`

Esto evita rellenar todos los tweets hasta 128 cuando son mucho más cortos.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=(
        8 if torch.cuda.is_available()
        else None
    ),
)

print("Tokenizer:", tokenizer.__class__.__name__)
print("Vocab size:", tokenizer.vocab_size)
print("Model max length:", tokenizer.model_max_length)


## 6. Dataset tokenizado

La tokenización no aprende parámetros del corpus, por lo que puede precalcularse para BASE y SOCIAL.

Las particiones train/validation/test de CV se aplican **después** mediante índices.


In [ ]:
class EncodedDataset(Dataset):
    def __init__(
        self,
        texts,
        labels,
        tokenizer,
        max_length=128,
    ):
        self.encodings = tokenizer(
            list(texts),
            truncation=True,
            max_length=max_length,
            padding=False,
        )

        self.labels = (
            np.asarray(
                labels,
                dtype=np.int64,
            )
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {
            key: values[idx]
            for key, values
            in self.encodings.items()
        }

        item["labels"] = int(
            self.labels[idx]
        )

        return item


def preparar_cache_datasets(df):
    y = (
        df[LABEL_COL]
        .to_numpy(dtype=np.int64)
    )

    cache = {}

    for prep, columna in TEXT_COLUMNS.items():
        cache[prep] = EncodedDataset(
            texts=df[columna].tolist(),
            labels=y,
            tokenizer=tokenizer,
            max_length=MAX_LENGTH,
        )

    return cache, y


## 7. Métricas

La métrica principal de selección es **F1-Macro**.

También se calculan accuracy, precision macro y recall macro como métricas secundarias.


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    if isinstance(logits, tuple):
        logits = logits[0]

    pred = np.argmax(
        logits,
        axis=-1,
    )

    return {
        "f1_macro": f1_score(
            labels,
            pred,
            average="macro",
        ),
        "accuracy": accuracy_score(
            labels,
            pred,
        ),
        "precision_macro": precision_score(
            labels,
            pred,
            average="macro",
            zero_division=0,
        ),
        "recall_macro": recall_score(
            labels,
            pred,
            average="macro",
            zero_division=0,
        ),
    }


## 8. Pérdida estándar o balanceada

Como IroSvA tiene desbalance aproximado 2:1, se deja que el Inner CV compare:

```text
standard
vs.
balanced
```

Cuando se usa `balanced`, los pesos se calculan **solo con las etiquetas del train del fold correspondiente**.  
La validación y el Outer Test nunca intervienen en esos pesos.


In [ ]:
def calcular_pesos_clase(y_train):
    clases = np.array(
        sorted(np.unique(y_train))
    )

    pesos = compute_class_weight(
        class_weight="balanced",
        classes=clases,
        y=y_train,
    )

    vector = np.ones(
        int(clases.max()) + 1,
        dtype=np.float32,
    )

    for clase, peso in zip(
        clases,
        pesos,
    ):
        vector[int(clase)] = peso

    return torch.tensor(
        vector,
        dtype=torch.float32,
    )


class WeightedTrainer(Trainer):
    def __init__(
        self,
        *args,
        class_weights=None,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
        **kwargs,
    ):
        labels = inputs.get("labels")
        outputs = model(**inputs)

        if self.class_weights is None:
            loss = outputs.loss
        else:
            logits = outputs.logits

            weights = self.class_weights.to(
                logits.device
            )

            loss = F.cross_entropy(
                logits,
                labels,
                weight=weights,
            )

        return (
            (loss, outputs)
            if return_outputs
            else loss
        )


## 9. Creación de BETO

Cada llamada crea un **modelo nuevo desde el checkpoint original**.

Esto garantiza independencia entre:

- trials;
- Inner Folds;
- Outer Folds;
- semillas finales.


In [ ]:
CLASSIFIER_DROPOUT = 0.1


def crear_modelo(seed):
    fijar_semilla(seed)
    # Guardar el informe de pesos (MISSING / UNEXPECTED) fuera de Jupyter.
    # Las capas de clasificación nuevas son normales al adaptar BETO a IroSvA.
    model_log = RUN_DIR / "carga_modelos.log"
    with model_log.open("a", encoding="utf-8") as log, redirect_stdout(log), redirect_stderr(log):
        config = AutoConfig.from_pretrained(MODEL_NAME, num_labels=2)
        config.classifier_dropout = CLASSIFIER_DROPOUT
        model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME, config=config
        )
    return model


## 10. TrainingArguments compatibles con distintas versiones de Transformers

El notebook detecta si tu versión usa `eval_strategy` o el nombre anterior `evaluation_strategy`.

Durante Inner CV:

- evaluación por época;
- guardado por época;
- `load_best_model_at_end=True`;
- mejor checkpoint seleccionado por `f1_macro`;
- early stopping con paciencia 1.

Durante refit:

- no se usa conjunto de validación;
- se entrena un número de épocas derivado de los Inner Folds.


In [ ]:
def crear_training_args(
    output_dir,
    learning_rate,
    batch_size,
    num_train_epochs,
    weight_decay,
    seed,
    con_evaluacion=True,
):
    params = {
        "output_dir": str(output_dir),
        "learning_rate": learning_rate,
        "per_device_train_batch_size": batch_size,
        "per_device_eval_batch_size": batch_size,
        "num_train_epochs": num_train_epochs,
        "weight_decay": weight_decay,
        "warmup_ratio": 0.10,
        "logging_strategy": "no",
        "save_total_limit": 1,
        "report_to": "none",
        "seed": seed,
        "data_seed": seed,
        "bf16": USE_BF16,
        "fp16": USE_FP16,
        "dataloader_num_workers": 0,
        "optim": "adamw_torch",
        "disable_tqdm": True,
    }

    if con_evaluacion:
        params.update({
            "save_strategy": "epoch",
            "load_best_model_at_end": True,
            "metric_for_best_model": "f1_macro",
            "greater_is_better": True,
        })
        eval_value = "epoch"
    else:
        params.update({
            "save_strategy": "no",
            "load_best_model_at_end": False,
        })
        eval_value = "no"

    firma = inspect.signature(
        TrainingArguments.__init__
    )

    if "eval_strategy" in firma.parameters:
        params["eval_strategy"] = eval_value
    elif "evaluation_strategy" in firma.parameters:
        params["evaluation_strategy"] = eval_value

    # Solo pasar argumentos soportados por la versión instalada.
    params = {
        k: v
        for k, v in params.items()
        if k in firma.parameters
    }

    return TrainingArguments(**params)


## 11. Espacio de búsqueda de Optuna

Se mantiene pequeño para que Nested CV sea viable.

```text
preprocessing    BASE / SOCIAL
learning_rate    2e-5 / 3e-5 / 5e-5
batch_size       16 / 32
epochs máximas   3 / 4 / 5
weight_decay     0 / 0.01 / 0.05
loss             standard / balanced
```

`max_length=128`, warmup=10% y dropout=0.1 permanecen fijos inicialmente.


In [ ]:
def sugerir_configuracion(trial):
    return {
        "preprocessing": trial.suggest_categorical(
            "preprocessing",
            ["BASE", "SOCIAL"],
        ),
        "learning_rate": trial.suggest_categorical(
            "learning_rate",
            [2e-5, 3e-5, 5e-5],
        ),
        "batch_size": trial.suggest_categorical(
            "batch_size",
            [16, 32],
        ),
        "num_train_epochs": trial.suggest_int(
            "num_train_epochs",
            3,
            5,
        ),
        "weight_decay": trial.suggest_categorical(
            "weight_decay",
            [0.0, 0.01, 0.05],
        ),
        "loss_mode": trial.suggest_categorical(
            "loss_mode",
            ["standard", "balanced"],
        ),
    }


## 12. Utilidades de entrenamiento

La mejor época de cada Inner Fold se recupera del historial de evaluación.

Para el refit del Outer Fold se utiliza la **mediana redondeada** de las mejores épocas internas.


In [ ]:
def obtener_mejor_epoca(trainer, max_epochs):
    candidatos = [
        x
        for x in trainer.state.log_history
        if "eval_f1_macro" in x
        and "epoch" in x
    ]

    if not candidatos:
        return int(max_epochs)

    mejor = max(
        candidatos,
        key=lambda x: x[
            "eval_f1_macro"
        ],
    )

    epoch = mejor.get(
        "epoch",
        max_epochs,
    )

    return max(
        1,
        min(
            int(max_epochs),
            int(round(float(epoch))),
        ),
    )


def predicciones_trainer(
    trainer,
    dataset,
):
    salida = trainer.predict(
        dataset
    )

    logits = salida.predictions

    if isinstance(logits, tuple):
        logits = logits[0]

    return np.argmax(
        logits,
        axis=-1,
    )


## Panel de seguimiento y recuperación

Se crea **un solo panel** al iniciar Nested CV. No se generan cientos de barras. Para comprobar el estado desde otra sesión de VS Code puedes abrir `../data/resultados_beto/seguimiento/<modo>/monitor.log`.

El panel muestra la etapa actual, entrenamientos completados y tiempo transcurrido. Si pasa cierto tiempo durante un entrenamiento, el callback emite una señal de actividad sin inundar la celda de salida. También se graba un resumen al completar **cada trial y cada Outer Fold**.

**Recuperación:** en caso de cierre/interrupción, vuelve a abrir **esta versión** y ejecuta desde el principio con el mismo `MODO`, mismos datasets y misma configuración. Los estudios SQLite guardan los trials terminados; los Outer Folds con CSV ya escrito no se repiten. Los resultados de ejecuciones hechas con el notebook antiguo no pueden importarse automáticamente, pues no utilizaba persistencia por trial.

In [ ]:
# =============================================================
# UN SOLO PANEL VIVO + LOG EN DISCO
# =============================================================
class MonitorBETO:
    def __init__(self, carpeta, total_estimado, ya_completados=0):
        self.carpeta = Path(carpeta)
        self.carpeta.mkdir(parents=True, exist_ok=True)
        self.log_path = self.carpeta / "monitor.log"
        self.inicio = time.monotonic()
        self.total = int(total_estimado)
        self.hechos = int(ya_completados)
        self.hechos_inicio = int(ya_completados)
        self.etapa = "Preparando experimento"
        self.ultima_senal = 0.0
        self.handle = display(HTML("<b>Preparando BETO...</b>"), display_id=True)
        self.registrar(f"Inicio: completados recuperados={self.hechos}/{self.total}")
        self.actualizar("Inicializando", forzar=True)

    def registrar(self, texto):
        fecha = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        with self.log_path.open("a", encoding="utf-8") as f:
            f.write(f"[{fecha}] {texto}\n")

    def actualizar(self, etapa=None, forzar=False):
        if etapa is not None:
            self.etapa = str(etapa)
        ahora = time.monotonic()
        if not forzar and ahora - self.ultima_senal < HEARTBEAT_SEGUNDOS:
            return
        self.ultima_senal = ahora
        transcurrido = max(0.0, ahora - self.inicio)
        realizados = max(0, self.hechos - self.hechos_inicio)
        faltan = max(0, self.total - self.hechos)
        # ETA solo después de observar al menos 2 fits completos en esta sesión.
        if realizados >= 2:
            restante = transcurrido / realizados * faltan
            eta = f"~{restante / 60:.0f} min (orientativo)"
        else:
            eta = "calculando..."
        progreso = 100 * self.hechos / max(1, self.total)
        fecha = datetime.now().strftime("%H:%M:%S")
        contenido = (
            '<div style="font-family:system-ui;padding:12px;border:1px solid #777;'
            'border-radius:8px;max-width:800px">'
            '<b>BETO — seguimiento en vivo</b><br>'
            f'Etapa: <b>{html.escape(self.etapa)}</b><br>'
            f'Entrenamientos completados: {self.hechos}/{self.total} '
            f'({progreso:.1f}%)<br>'
            f'<progress value="{self.hechos}" max="{max(self.total,1)}" '
            'style="width:100%"></progress><br>'
            f'Tiempo de esta sesión: {transcurrido / 60:.1f} min '
            f'| Tiempo restante: {eta}<br>'
            f'Última señal de actividad: {fecha}<br>'
            f'Log: {html.escape(str(self.log_path))}'
            '</div>'
        )
        self.handle.update(HTML(contenido))

    def iniciar_etapa(self, etapa):
        self.registrar(etapa)
        self.actualizar(etapa, forzar=True)

    def pulso(self):
        # Además de actualizar el panel, deja una señal en el log cada 30 s
        # para poder comprobar actividad desde OTRA ventana de VS Code.
        if time.monotonic() - self.ultima_senal >= HEARTBEAT_SEGUNDOS:
            self.registrar(f"ACTIVO: {self.etapa}")
        self.actualizar(forzar=False)

    def terminar_fit(self, texto="Entrenamiento finalizado"):
        self.hechos += 1
        self.registrar(texto)
        self.actualizar(texto, forzar=True)


class PulsoTrainer(TrainerCallback):
    def __init__(self, monitor):
        self.monitor = monitor

    def on_step_end(self, args, state, control, **kwargs):
        if self.monitor is not None:
            self.monitor.pulso()
        return control


def silenciar_trainer(trainer):
    # Con disable_tqdm=True Trainer instala PrinterCallback, que puede
    # seguir imprimiendo todas las métricas por época. Retirarlo también.
    trainer.remove_callback(PrinterCallback)
    trainer.remove_callback(ProgressCallback)
    return trainer


def guardar_csv_atomico(df, ruta):
    ruta = Path(ruta)
    temporal = ruta.with_name(ruta.name + ".tmp")
    df.to_csv(temporal, index=False)
    os.replace(temporal, ruta)


def firma_experimento():
    """Evita reanudar por error con datos o parámetros diferentes."""
    dataset = {}
    for variante in VARIANTES:
        ruta = DATA_DIR / f"train_transformers_{variante}.csv"
        if not ruta.exists():
            raise FileNotFoundError(ruta)
        dataset[variante] = hashlib.sha256(ruta.read_bytes()).hexdigest()
    return {
        "model_name": MODEL_NAME,
        "modo": MODO,
        "variantes": VARIANTES,
        "outer": N_OUTER,
        "inner": N_INNER,
        "trials": N_TRIALS,
        "max_length": MAX_LENGTH,
        "seed": RANDOM_STATE,
        "dropout": CLASSIFIER_DROPOUT,
        "texto": TEXT_COLUMNS,
        "datasets_sha256": dataset,
    }


def comprobar_firma_experimento():
    ruta = RUN_DIR / "config_ejecucion.json"
    actual = firma_experimento()
    if ruta.exists():
        anterior = json.loads(ruta.read_text(encoding="utf-8"))
        if anterior != actual:
            raise ValueError(
                "Ya hay una corrida anterior con configuración/datos distintos "
                f"en {RUN_DIR}. No mezcles resultados; cambia de carpeta."
            )
    else:
        temporal = ruta.with_suffix(".tmp")
        temporal.write_text(json.dumps(actual, indent=2), encoding="utf-8")
        os.replace(temporal, ruta)


def ruta_estudio(variante, etiqueta):
    return RUN_DIR / f"beto_{variante}_{etiqueta}.db"


def cargar_estudio_si_existe(variante, etiqueta):
    ruta = ruta_estudio(variante, etiqueta)
    if not ruta.exists():
        return None
    return optuna.load_study(
        study_name=f"BETO_{variante}_{etiqueta}",
        storage=f"sqlite:///{ruta.resolve().as_posix()}",
    )


def contar_avance_guardado():
    """Estimación de fits terminados; no incluye trials interrumpidos a medias."""
    hechos = 0
    for variante in VARIANTES:
        parciales = RUN_DIR / f"beto_nestedcv_detalle_{variante}.csv"
        completados = set()
        if parciales.exists():
            df = pd.read_csv(parciales)
            if "outer_fold" in df:
                completados = set(df["outer_fold"].astype(int))
        for outer in range(1, N_OUTER + 1):
            if outer in completados:
                hechos += N_TRIALS * N_INNER + 1
            else:
                try:
                    study = cargar_estudio_si_existe(variante, f"outer{outer}")
                    if study is not None:
                        hechos += min(
                            N_TRIALS,
                            sum(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials),
                        ) * N_INNER
                except (KeyError, ValueError):
                    pass
    return hechos


## 13. Optimización Inner CV

Esta función puede utilizarse tanto:

- dentro de cada Outer Fold;
- como optimización final sobre todo el train.

Cada trial ejecuta todos los Inner Folds y devuelve el **F1-Macro promedio**.


In [ ]:
def optimizar_en_indices(
    df,
    cache,
    y,
    indices_base,
    variante,
    etiqueta_busqueda,
    n_inner,
    n_trials,
    seed_cv,
    monitor=None,
):
    indices_base = np.asarray(indices_base, dtype=int)
    y_base = y[indices_base]
    inner_cv = StratifiedKFold(n_splits=n_inner, shuffle=True, random_state=seed_cv)
    splits = list(inner_cv.split(np.zeros(len(indices_base)), y_base))

    def objective(trial):
        cfg = sugerir_configuracion(trial)
        dataset_completo = cache[cfg["preprocessing"]]
        fold_scores, fold_epochs = [], []
        for inner_fold, (idx_train_local, idx_val_local) in enumerate(splits, start=1):
            idx_train = indices_base[idx_train_local]
            idx_val = indices_base[idx_val_local]
            if monitor:
                monitor.iniciar_etapa(
                    f"{variante.upper()} | {etiqueta_busqueda} | "
                    f"trial id={trial.number} (meta {n_trials} completos) | inner {inner_fold}/{n_inner}"
                )
            train_ds = Subset(dataset_completo, idx_train.tolist())
            val_ds = Subset(dataset_completo, idx_val.tolist())
            seed_modelo = RANDOM_STATE + seed_cv * 10 + trial.number * 100 + inner_fold
            model = crear_modelo(seed_modelo)
            class_weights = (
                calcular_pesos_clase(y[idx_train]) if cfg["loss_mode"] == "balanced" else None
            )
            with tempfile.TemporaryDirectory(dir=TMP_DIR) as temp_dir:
                args = crear_training_args(
                    output_dir=temp_dir,
                    learning_rate=cfg["learning_rate"],
                    batch_size=cfg["batch_size"],
                    num_train_epochs=cfg["num_train_epochs"],
                    weight_decay=cfg["weight_decay"],
                    seed=seed_modelo,
                    con_evaluacion=True,
                )
                callbacks = [EarlyStoppingCallback(early_stopping_patience=1)]
                if monitor:
                    callbacks.append(PulsoTrainer(monitor))
                trainer = silenciar_trainer(WeightedTrainer(
                    model=model,
                    args=args,
                    train_dataset=train_ds,
                    eval_dataset=val_ds,
                    data_collator=data_collator,
                    compute_metrics=compute_metrics,
                    callbacks=callbacks,
                    class_weights=class_weights,
                ))
                trainer.train()
                metrics = trainer.evaluate()
                score = float(metrics["eval_f1_macro"])
                best_epoch = obtener_mejor_epoca(trainer, cfg["num_train_epochs"])
            fold_scores.append(score)
            fold_epochs.append(best_epoch)
            del trainer, model
            liberar_memoria()
            if monitor:
                monitor.terminar_fit(
                    f"{variante.upper()} {etiqueta_busqueda} "
                    f"trial {trial.number+1} inner {inner_fold}: F1={score:.4f}"
                )

        std_score = float(np.std(fold_scores, ddof=1)) if len(fold_scores) > 1 else 0.0
        refit_epochs = int(np.clip(round(np.median(fold_epochs)), 1, cfg["num_train_epochs"]))
        trial.set_user_attr("inner_f1_std", std_score)
        trial.set_user_attr("best_epochs", [int(x) for x in fold_epochs])
        trial.set_user_attr("refit_epochs", refit_epochs)
        return float(np.mean(fold_scores))

    # Persistir los trials completos inmediatamente, antes de acabar el Outer.
    etiqueta = str(etiqueta_busqueda)
    db = ruta_estudio(variante, etiqueta)
    sampler_path = RUN_DIR / f"sampler_{variante}_{etiqueta}.pkl"
    if sampler_path.exists():
        with sampler_path.open("rb") as f:
            sampler = pickle.load(f)  # Solo archivos propios de esta corrida.
    else:
        sampler = optuna.samplers.TPESampler(seed=seed_cv)

    study = optuna.create_study(
        direction="maximize",
        sampler=sampler,
        study_name=f"BETO_{variante}_{etiqueta}",
        storage=f"sqlite:///{db.resolve().as_posix()}",
        load_if_exists=True,
    )
    completados_antes = sum(
        trial.state == optuna.trial.TrialState.COMPLETE for trial in study.trials
    )
    restantes = max(0, n_trials - completados_antes)
    if monitor and completados_antes:
        monitor.registrar(
            f"Reanudado {variante} {etiqueta}: "
            f"{completados_antes} trials ya completos, {restantes} por ejecutar"
        )

    def al_terminar_trial(estudio, trial):
        # El DB persiste el trial y este callback guarda el estado del sampler.
        temporal = sampler_path.with_suffix(".pkl.tmp")
        with temporal.open("wb") as f:
            pickle.dump(estudio.sampler, f)
        os.replace(temporal, sampler_path)
        estudio.trials_dataframe().to_csv(
            RUN_DIR / f"trials_{variante}_{etiqueta}.csv", index=False
        )
        completados = sum(
            t.state == optuna.trial.TrialState.COMPLETE for t in estudio.trials
        )
        if trial.value is not None:
            mensaje = (
                f"{variante.upper()} {etiqueta} | trial {completados}/{n_trials} "
                f"F1_inner={trial.value:.4f} | mejor={estudio.best_value:.4f}"
            )
        else:
            mensaje = f"{variante.upper()} {etiqueta} | trial {trial.number} falló"
        if monitor:
            monitor.registrar(mensaje)
            monitor.actualizar(mensaje, forzar=True)
        print(mensaje, flush=True)  # Máximo 10 líneas por Outer Fold.

    if restantes:
        study.optimize(
            objective,
            n_trials=restantes,
            callbacks=[al_terminar_trial],
            show_progress_bar=False,
            gc_after_trial=True,
        )
    if sum(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials) < n_trials:
        raise RuntimeError("La búsqueda no completó los trials previstos. Revisa el log.")
    return study


## 14. Refit y evaluación de un Outer Fold

Después del Inner CV:

1. se toma la mejor configuración;
2. se crea un **BETO nuevo**;
3. se entrena con todo el Outer Train;
4. se evalúa una única vez en el Outer Test.

El Outer Test nunca se usa para elegir configuración.


In [ ]:
def refit_outer_y_evaluar(cache, y, idx_train, idx_outer, best_params, refit_epochs, seed, monitor=None):
    prep = best_params["preprocessing"]
    dataset_completo = cache[prep]
    train_dataset = Subset(dataset_completo, list(map(int, idx_train)))
    outer_dataset = Subset(dataset_completo, list(map(int, idx_outer)))
    if monitor:
        monitor.iniciar_etapa(f"Refit externo | {prep} | {refit_epochs} épocas")
    model = crear_modelo(seed)
    class_weights = (
        calcular_pesos_clase(y[idx_train]) if best_params["loss_mode"] == "balanced" else None
    )
    with tempfile.TemporaryDirectory(dir=TMP_DIR) as temp_dir:
        args = crear_training_args(
            output_dir=temp_dir,
            learning_rate=best_params["learning_rate"],
            batch_size=best_params["batch_size"],
            num_train_epochs=refit_epochs,
            weight_decay=best_params["weight_decay"],
            seed=seed,
            con_evaluacion=False,
        )
        callbacks = [PulsoTrainer(monitor)] if monitor else []
        trainer = silenciar_trainer(WeightedTrainer(
            model=model,
            args=args,
            train_dataset=train_dataset,
            data_collator=data_collator,
            callbacks=callbacks,
            class_weights=class_weights,
        ))
        trainer.train()
        pred_train = predicciones_trainer(trainer, train_dataset)
        pred_outer = predicciones_trainer(trainer, outer_dataset)
    f1_train = f1_score(y[idx_train], pred_train, average="macro")
    f1_outer = f1_score(y[idx_outer], pred_outer, average="macro")
    del trainer, model
    liberar_memoria()
    if monitor:
        monitor.terminar_fit(f"Refit Outer completado: F1_train={f1_train:.4f}, F1_outer={f1_outer:.4f}")
    return {
        "train_f1_macro": float(f1_train),
        "outer_f1_macro": float(f1_outer),
        "generalization_gap": float(f1_train-f1_outer),
    }


## 15. Nested CV 5×3

En modo completo:

```text
5 Outer Folds
× 10 trials
× 3 Inner Folds
= 150 fine-tunings internos por variante
```

más 5 refits externos.

No ejecutes el modo completo hasta haber validado primero `MODO="prueba"`.


In [ ]:
def ejecutar_nested_variante(variante, monitor=None):
    df = cargar_train_variante(variante)
    cache, y = preparar_cache_datasets(df)
    outer_cv = StratifiedKFold(n_splits=N_OUTER, shuffle=True, random_state=RANDOM_STATE)
    splits = list(outer_cv.split(np.zeros(len(y)), y))
    ruta_parcial = RUN_DIR / f"beto_nestedcv_detalle_{variante}.csv"
    if ruta_parcial.exists():
        resultados = pd.read_csv(ruta_parcial).to_dict("records")
        completados = {int(fila["outer_fold"]) for fila in resultados}
    else:
        resultados, completados = [], set()
    studies = {}

    for outer_fold, (idx_train, idx_outer) in enumerate(splits, start=1):
        if outer_fold in completados:
            print(f"{variante.upper()} Outer {outer_fold}/{N_OUTER}: ya guardado, se omite.", flush=True)
            continue
        etiqueta = f"outer{outer_fold}"
        if monitor:
            monitor.iniciar_etapa(f"{variante.upper()} Outer {outer_fold}/{N_OUTER}: preparando búsqueda")
        study = optimizar_en_indices(
            df=df,
            cache=cache,
            y=y,
            indices_base=idx_train,
            variante=variante,
            etiqueta_busqueda=etiqueta,
            n_inner=N_INNER,
            n_trials=N_TRIALS,
            seed_cv=RANDOM_STATE + outer_fold,
            monitor=monitor,
        )
        studies[outer_fold] = study
        best_params = study.best_params.copy()
        refit_epochs = int(study.best_trial.user_attrs["refit_epochs"])
        resultado = refit_outer_y_evaluar(
            cache=cache,
            y=y,
            idx_train=idx_train,
            idx_outer=idx_outer,
            best_params=best_params,
            refit_epochs=refit_epochs,
            seed=RANDOM_STATE + 1000 + outer_fold,
            monitor=monitor,
        )
        fila = {
            "variante": variante,
            "outer_fold": outer_fold,
            "best_preprocessing": best_params["preprocessing"],
            "inner_best_f1": float(study.best_value),
            "inner_best_std": float(study.best_trial.user_attrs["inner_f1_std"]),
            "refit_epochs": refit_epochs,
            **resultado,
            "best_params": json.dumps(best_params, ensure_ascii=False),
        }
        resultados.append(fila)
        # Guardar ANTES de arrancar el siguiente Outer Fold.
        guardar_csv_atomico(pd.DataFrame(resultados).sort_values("outer_fold"), ruta_parcial)
        msg = (
            f"{variante.upper()} Outer {outer_fold}/{N_OUTER} FINALIZADO "
            f"| F1_inner={fila['inner_best_f1']:.4f} "
            f"| F1_train={fila['train_f1_macro']:.4f} "
            f"| F1_outer={fila['outer_f1_macro']:.4f} "
            f"| gap={fila['generalization_gap']:.4f}"
        )
        print(msg, flush=True)
        if monitor:
            monitor.registrar(msg)
            monitor.actualizar(msg, forzar=True)

    df_resultados = pd.DataFrame(resultados).sort_values("outer_fold").reset_index(drop=True)
    resumen = {
        "variante": variante,
        "n": len(y),
        "f1_train_mean": df_resultados["train_f1_macro"].mean(),
        "f1_nested_mean": df_resultados["outer_f1_macro"].mean(),
        "f1_nested_std": (
            df_resultados["outer_f1_macro"].std(ddof=1)
            if len(df_resultados) > 1 else 0.0
        ),
        "f1_nested_min": df_resultados["outer_f1_macro"].min(),
        "f1_nested_max": df_resultados["outer_f1_macro"].max(),
        "gap_mean": df_resultados["generalization_gap"].mean(),
    }
    return df_resultados, resumen, studies


## Ejecutar Nested CV con seguimiento en vivo

Primero usa `MODO = "prueba"` para verificar que la instalación funciona; luego cambia a `"completo"`, reinicia el kernel y ejecuta desde arriba. Verifica en la configuración: **MX/ES/CU, Outer 5, Inner 3, 10 trials**.

Esta celda muestra un único **panel vivo** y una línea al terminar cada trial y cada Outer Fold. El panel tiene hora de último movimiento y un ETA orientativo. Si VS Code oculta la salida, abre `../data/resultados_beto/seguimiento/completo/monitor.log` desde otra ventana para ver qué fold/trial se está ejecutando.

**Si la ejecución se interrumpe:** vuelve a ejecutar este notebook con los mismos datos y `MODO`; recupera los trials acabados desde SQLite y omite los Outer Folds guardados. La reproducción exacta de una ejecución interrumpida depende también del estado de los generadores aleatorios; se intenta conservar el sampler de Optuna, pero si se interrumpe durante un trial ese trial debe repetirse.


In [ ]:
# Un panel vivo para TODA la ejecución; los resultados se guardan por fold.
comprobar_firma_experimento()
fits_esperados = len(VARIANTES) * N_OUTER * (N_TRIALS * N_INNER + 1)
monitor = MonitorBETO(
    carpeta=RUN_DIR,
    total_estimado=fits_esperados,
    ya_completados=contar_avance_guardado(),
)

nested_detalle = {}
nested_resumen = []
nested_studies = {}

try:
    for variante in VARIANTES:
        detalle, resumen, studies = ejecutar_nested_variante(variante, monitor=monitor)
        nested_detalle[variante] = detalle
        nested_resumen.append(resumen)
        nested_studies[variante] = studies
        # Resumen acumulativo: aunque se interrumpa ES o CU, MX queda guardado.
        guardar_csv_atomico(pd.DataFrame(nested_resumen), RUN_DIR / "beto_nestedcv_resumen_parcial.csv")
except KeyboardInterrupt:
    monitor.registrar("INTERRUMPIDO. Los trials completos están en SQLite; Outer completos en CSV.")
    monitor.actualizar("Interrumpido: resultados parciales guardados", forzar=True)
    raise
except Exception as e:
    monitor.registrar(f"ERROR: {type(e).__name__}: {e}")
    monitor.actualizar(f"Error: {type(e).__name__}; revisar monitor.log", forzar=True)
    raise
else:
    monitor.registrar("Nested CV completo para todas las variantes.")
    monitor.actualizar("Completado; resultados guardados", forzar=True)

# Solo tres filas en el resumen completo y cinco por variante.
df_nested_resumen = pd.DataFrame(nested_resumen)
print("\nRESUMEN NESTED CV BETO")
display(df_nested_resumen)
for variante, df in nested_detalle.items():
    print(f"\nDETALLE {variante.upper()}")
    display(df[[
        "outer_fold", "best_preprocessing", "inner_best_f1",
        "inner_best_std", "refit_epochs", "train_f1_macro",
        "outer_f1_macro", "generalization_gap",
    ]])


## Guardar y consultar resultados

Los archivos se guardan *durante* la ejecución, sin esperar al último fold. `beto_nestedcv_detalle_<variante>.csv` registra cada Outer Fold completado y `beto_<variante>_outer<fold>.db` conserva los trials de Optuna. Esta celda, que se ejecuta solo al terminar las tres variantes, consolida los archivos definitivos.


In [ ]:
# La sección anterior ya guardó trials y Outer Folds al completarse.
# Aquí solo se consolidan los archivos finales si toda la corrida terminó.
if MODO == "completo":
    guardar_csv_atomico(df_nested_resumen, RESULTS_DIR / "beto_nestedcv_resumen.csv")
    for variante in VARIANTES:
        detalle = nested_detalle[variante].copy()
        guardar_csv_atomico(
            detalle, RESULTS_DIR / f"beto_nestedcv_detalle_{variante}.csv"
        )
        for outer_fold, study in nested_studies[variante].items():
            study.trials_dataframe().to_csv(
                RESULTS_DIR / f"beto_optuna_{variante}_outer{outer_fold}.csv",
                index=False,
            )
    print("Archivos definitivos guardados. Los parciales siguen en:", RUN_DIR)
else:
    print("Modo prueba: resultados aislados en:", RUN_DIR)


## 18. Optimización final sobre todo el train

**No es necesario ejecutar esta sección inmediatamente.**

Primero revisa los resultados Nested CV.

Cuando la metodología quede congelada:

```python
EJECUTAR_OPTIMIZACION_FINAL = True
```

La búsqueda final vuelve a realizar Inner CV de 3 folds sobre **todo el train**.  
El test oficial sigue sin cargarse.


In [ ]:
studies_finales = {}
configs_finales = {}

if EJECUTAR_OPTIMIZACION_FINAL:
    if MODO != "completo":
        raise ValueError(
            "La optimización final solo "
            "debe ejecutarse en MODO='completo'."
        )

    for variante in VARIANTES:
        print(
            f"\nOPTIMIZACIÓN FINAL — "
            f"{variante.upper()}"
        )

        df = cargar_train_variante(
            variante
        )

        cache, y = (
            preparar_cache_datasets(
                df
            )
        )

        indices = np.arange(
            len(y)
        )

        study = optimizar_en_indices(
            df=df,
            cache=cache,
            y=y,
            indices_base=indices,
            variante=variante,
            etiqueta_busqueda="final",
            n_inner=3,
            n_trials=N_TRIALS,
            seed_cv=RANDOM_STATE,
            monitor=None,
        )

        studies_finales[
            variante
        ] = study

        cfg = (
            study.best_params.copy()
        )

        cfg[
            "refit_epochs"
        ] = int(
            study.best_trial.user_attrs[
                "refit_epochs"
            ]
        )

        configs_finales[
            variante
        ] = cfg

        print(
            "Mejor F1-CV:",
            f"{study.best_value:.4f}",
        )
        print(
            "Configuración:",
            cfg,
        )

        study.trials_dataframe().to_csv(
            RESULTS_DIR
            / (
                "beto_optuna_final_"
                f"{variante}.csv"
            ),
            index=False,
        )

    with open(
        RESULTS_DIR
        / "beto_config_final.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            configs_finales,
            f,
            indent=4,
            ensure_ascii=False,
        )

    print(
        "\nConfiguraciones finales guardadas."
    )
else:
    print(
        "Optimización final bloqueada. "
        "Revisa primero Nested CV."
    )


## 19. TEST OFICIAL — BLOQUEADO POR DEFECTO

Esta sección debe ejecutarse **solo cuando**:

1. Nested CV ya fue revisado;
2. no se cambiará preprocessing;
3. no se cambiará espacio de hiperparámetros;
4. la optimización final ya terminó;
5. la metodología quedó congelada.

Entonces:

```python
EJECUTAR_TEST_OFICIAL = True
```

Se entrenan **5 modelos finales independientes**, uno por semilla, con la configuración congelada.  
Se reporta **media ± desviación** del F1-Macro del test; nunca se selecciona la mejor semilla.


In [ ]:
def cargar_test_variante(
    variante,
):
    ruta = (
        DATA_DIR
        / f"test_transformers_{variante}.csv"
    )

    if not ruta.exists():
        raise FileNotFoundError(
            ruta
        )

    df = pd.read_csv(ruta)

    df[LABEL_COL] = (
        df[LABEL_COL]
        .astype(int)
    )

    for col in TEXT_COLUMNS.values():
        df[col] = (
            df[col]
            .fillna("")
            .astype(str)
        )

    return df.reset_index(
        drop=True
    )


def entrenar_final_una_semilla(
    df_train,
    df_test,
    cfg,
    seed,
):
    prep = cfg[
        "preprocessing"
    ]

    train_dataset = EncodedDataset(
        texts=df_train[
            TEXT_COLUMNS[prep]
        ].tolist(),
        labels=df_train[
            LABEL_COL
        ].to_numpy(),
        tokenizer=tokenizer,
        max_length=MAX_LENGTH,
    )

    test_dataset = EncodedDataset(
        texts=df_test[
            TEXT_COLUMNS[prep]
        ].tolist(),
        labels=df_test[
            LABEL_COL
        ].to_numpy(),
        tokenizer=tokenizer,
        max_length=MAX_LENGTH,
    )

    y_train = (
        df_train[LABEL_COL]
        .to_numpy(dtype=np.int64)
    )

    y_test = (
        df_test[LABEL_COL]
        .to_numpy(dtype=np.int64)
    )

    class_weights = None

    if cfg[
        "loss_mode"
    ] == "balanced":
        class_weights = (
            calcular_pesos_clase(
                y_train
            )
        )

    model = crear_modelo(
        seed
    )

    with tempfile.TemporaryDirectory(
        dir=TMP_DIR
    ) as temp_dir:
        args = crear_training_args(
            output_dir=temp_dir,
            learning_rate=cfg[
                "learning_rate"
            ],
            batch_size=cfg[
                "batch_size"
            ],
            num_train_epochs=cfg[
                "refit_epochs"
            ],
            weight_decay=cfg[
                "weight_decay"
            ],
            seed=seed,
            con_evaluacion=False,
        )

        trainer = silenciar_trainer(WeightedTrainer(
            model=model,
            args=args,
            train_dataset=train_dataset,
            data_collator=data_collator,
            class_weights=class_weights,
        ))

        trainer.train()

        pred = predicciones_trainer(
            trainer,
            test_dataset,
        )

    resultado = {
        "seed": seed,
        "f1_macro": f1_score(
            y_test,
            pred,
            average="macro",
        ),
        "accuracy": accuracy_score(
            y_test,
            pred,
        ),
        "precision_macro": precision_score(
            y_test,
            pred,
            average="macro",
            zero_division=0,
        ),
        "recall_macro": recall_score(
            y_test,
            pred,
            average="macro",
            zero_division=0,
        ),
    }

    del trainer
    del model
    liberar_memoria()

    return resultado


In [ ]:
if EJECUTAR_TEST_OFICIAL:
    if not EJECUTAR_OPTIMIZACION_FINAL:
        raise ValueError(
            "Primero ejecuta la optimización final."
        )

    resultados_test = []

    for variante in VARIANTES:
        df_train = cargar_train_variante(
            variante
        )

        df_test = cargar_test_variante(
            variante
        )

        cfg = configs_finales[
            variante
        ]

        print(
            f"\nTEST OFICIAL — "
            f"{variante.upper()}"
        )
        print(
            "Configuración congelada:",
            cfg,
        )

        for seed in SEMILLAS_FINALES:
            resultado = (
                entrenar_final_una_semilla(
                    df_train=df_train,
                    df_test=df_test,
                    cfg=cfg,
                    seed=seed,
                )
            )

            resultado[
                "variante"
            ] = variante

            resultados_test.append(
                resultado
            )

            print(
                f"Seed {seed}: "
                f"F1={resultado['f1_macro']:.4f}"
            )

    df_test_final = pd.DataFrame(
        resultados_test
    )

    resumen_test = (
        df_test_final
        .groupby("variante")
        .agg(
            f1_mean=(
                "f1_macro",
                "mean",
            ),
            f1_std=(
                "f1_macro",
                "std",
            ),
            accuracy_mean=(
                "accuracy",
                "mean",
            ),
            precision_macro_mean=(
                "precision_macro",
                "mean",
            ),
            recall_macro_mean=(
                "recall_macro",
                "mean",
            ),
        )
        .reset_index()
    )

    print(
        "\nRESULTADO FINAL BETO"
    )

    display(
        resumen_test.style.format({
            "f1_mean": "{:.4f}",
            "f1_std": "{:.4f}",
            "accuracy_mean": "{:.4f}",
            "precision_macro_mean": "{:.4f}",
            "recall_macro_mean": "{:.4f}",
        })
    )

    df_test_final.to_csv(
        RESULTS_DIR
        / "beto_test_5_seeds.csv",
        index=False,
    )

    resumen_test.to_csv(
        RESULTS_DIR
        / "beto_test_resumen.csv",
        index=False,
    )
else:
    print(
        "TEST OFICIAL BLOQUEADO. "
        "No se ha cargado ni evaluado."
    )


## 20. Qué revisar después de Nested CV

Antes de tocar el test oficial, revisar por variante:

1. `f1_nested_mean ± f1_nested_std`;
2. `f1_train_mean`;
3. `gap_mean`;
4. BASE vs SOCIAL elegidos en los Outer Folds;
5. `standard` vs `balanced`;
6. learning rates seleccionados;
7. estabilidad de las mejores épocas;
8. comportamiento de los 10 trials de Optuna.

Solo después se decide si el espacio de búsqueda está suficientemente bien planteado.

### Interpretación rápida

```text
Train muy alto + Outer bastante menor
→ overfitting

Train y Outer ambos bajos
→ underfitting / configuración insuficiente

Inner alto + Outer claramente menor
→ posible selección demasiado optimista

Outer estable entre folds
→ buena señal de generalización
```


## 21. Referencias metodológicas implementadas

- BETO: checkpoint monolingüe español `dccuchile/bert-base-spanish-wwm-cased`.
- Hugging Face Trainer: evaluación/guardado por época, `load_best_model_at_end` y métrica `f1_macro`.
- Optuna: cada configuración se evalúa con modelos recién inicializados.
- La búsqueda usa F1-Macro como objetivo, no `eval_loss`.
- El test oficial permanece aislado de la selección.


### Documentación de control y recuperación

- [Callbacks de Hugging Face](https://huggingface.co/docs/transformers/main_classes/callback): `disable_tqdm`, `PrinterCallback`, `EarlyStoppingCallback` y callbacks para seguimiento.
- [Persistencia con SQLite en Optuna](https://optuna.readthedocs.io/en/stable/tutorial/20_recipes/001_rdb.html): uso de `storage` y `load_if_exists=True` para recuperar estudios terminados.
- **No reanuda la corrida anterior** realizada con `02_beto_nestedcv_5x3.ipynb`, porque ese archivo no escribía su historial de Optuna en SQLite. Si el kernel original todavía está vivo y la celda acabó, conserva sus tablas exportándolas antes de reiniciarlo.
